In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1430_Rohini_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,192.93,279.54,3.37,17.22,20.57,57.67,3.85,0.72,9.57,...,NaN,8.05,76.12,0.81,264.86,0.00,0.00,59.09,978.60,NaN
1,2024-01-02,194.73,276.34,5.58,20.14,25.73,45.07,5.25,0.87,9.43,...,NaN,7.72,72.79,0.84,293.63,0.00,0.00,83.26,977.64,NaN
2,2024-01-03,228.73,325.08,21.96,24.44,46.63,47.26,2.97,1.82,8.06,...,NaN,6.97,83.90,0.75,249.00,0.00,0.00,46.22,977.85,NaN
3,2024-01-04,225.10,311.02,9.35,24.78,25.63,42.46,3.62,1.34,1.92,...,NaN,6.67,85.57,0.84,249.44,0.00,0.00,22.12,978.20,NaN
4,2024-01-05,159.96,261.34,5.86,30.03,20.83,44.49,6.43,1.13,8.95,...,NaN,8.19,87.36,0.84,263.44,0.00,0.00,15.85,978.55,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,158.08,192.12,10.37,63.65,42.29,54.70,5.44,1.21,25.12,...,NaN,15.72,92.79,1.65,151.04,0.58,0.42,9.64,990.30,NaN
362,2024-12-28,104.08,129.75,11.92,62.63,43.20,45.96,6.16,1.18,13.15,...,NaN,16.34,94.18,1.27,202.45,0.04,0.04,11.28,989.73,NaN
363,2024-12-29,94.88,120.71,3.67,32.96,20.53,30.57,6.70,0.78,27.45,...,NaN,15.06,89.91,1.54,252.26,0.00,0.00,47.78,991.07,NaN
364,2024-12-30,105.21,131.75,3.85,33.89,21.16,25.58,6.02,0.79,37.74,...,NaN,13.17,84.65,1.52,244.62,0.00,0.00,48.32,990.41,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         192.93        279.54        3.37        17.22   
1  2024-01-02         194.73        276.34        5.58        20.14   
2  2024-01-03         228.73        325.08       21.96        24.44   
3  2024-01-04         225.10        311.02        9.35        24.78   
4  2024-01-05         159.96        261.34        5.86        30.03   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      20.57        57.67         3.85        0.72           9.57   
1      25.73        45.07         5.25        0.87           9.43   
2      46.63        47.26         2.97        1.82           8.06   
3      25.63        42.46         3.62        1.34           1.92   
4      20.83        44.49         6.43        1.13           8.95   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0            0.455            15.71     8.05   76.12      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.165494,0.372731,-0.967211,-1.345813,-0.813370,0.852038,-1.237645,-0.668731,-1.194144,-0.175863,-0.014495,-2.146549,0.849298,-0.613576,1.326918,0.0,0.0,-1.217412,-0.921020
1,2024-01-02,1.189046,0.345268,-0.852990,-1.182159,-0.599410,-0.128001,-0.973632,-0.421137,-1.200479,-0.175863,-0.014495,-2.186090,0.632768,-0.520471,1.847466,0.0,0.0,-0.752174,-1.060666
2,2024-01-03,1.633928,0.763570,-0.006414,-0.941161,0.267211,0.042340,-1.403596,1.146964,-1.262473,-0.175863,-0.014495,-2.275954,1.355185,-0.799785,1.039956,0.0,0.0,-1.465141,-1.030118
3,2024-01-04,1.586430,0.642903,-0.658143,-0.922105,-0.603556,-0.331009,-1.281019,0.354660,-1.540315,-0.175863,-0.014495,-2.311900,1.463775,-0.520471,1.047917,0.0,0.0,-1.929032,-0.979206
4,2024-01-05,0.734089,0.216533,-0.838519,-0.627864,-0.802589,-0.173113,-0.751107,0.008028,-1.222200,-0.175863,-0.014495,-2.129775,1.580168,-0.520471,1.301225,0.0,0.0,-2.049721,-0.928293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.709490,-0.377536,-0.605426,1.256405,0.087252,0.621029,-0.937802,0.140078,-0.490489,-0.848573,-0.783528,-1.227537,1.933249,1.993358,-0.732476,0.0,0.0,-2.169254,0.780906
362,2024-12-28,0.002913,-0.912815,-0.525316,1.199238,0.124986,-0.058776,-0.802024,0.090559,-1.032145,-0.907070,-0.845882,-1.153249,2.023632,0.814031,0.197707,0.0,0.0,-2.137687,0.697992
363,2024-12-29,-0.117466,-0.990400,-0.951705,-0.463649,-0.815028,-1.255823,-0.700190,-0.569694,-0.385053,-0.829074,-1.243389,-1.306617,1.745979,1.651973,1.098941,0.0,0.0,-1.435113,0.892913
364,2024-12-30,0.017699,-0.895651,-0.942402,-0.411526,-0.788905,-1.643950,-0.828425,-0.553187,0.080581,-0.790076,-1.314836,-1.533076,1.403953,1.589904,0.960707,0.0,0.0,-1.424719,0.796907


In [10]:
df.to_excel("rohini2024.xlsx", index=False)